In [1]:
# Add project root to sys.path for imports
import sys
from pathlib import Path

# Add project root to sys.path
sys.path.append(str(Path.cwd().parent))

# Project Title
## Overview

### ICSID Worldbank data


Only use the cell below to download data if needed.  Cell keeped for transparency, reproducibility and reference.

In [2]:
## Download all ICSID cases from url.  Also retrive only mining cases (es = 101)

# import requests
# import pandas as pd

# url = "https://icsid.worldbank.org/api/all/cases"

# params = {
#     "es": 101
# }

# r = requests.get(url)
# r_mining = requests.get(url, params=params)

# data = r.json()
# data_mining = r_mining.json()

# cases = data["data"]["GetAllCasesResult"]
# cases_mining = data_mining["data"]["GetAllCasesResult"]

# df = pd.json_normalize(cases)
# df_mining = pd.json_normalize(cases_mining)

## Save raw dataframes to csv files for documentation

# df.to_csv(r"../data/raw/icsid_cases_all_202605.csv", index=False)
#df_mining.to_csv(r"../data/raw/icsid_cases_mining_202605.csv", index=False)


In [3]:
import pandas as pd
df_raw = pd.read_csv(r"../data/raw/icsid_cases_all_202605.csv")
df = df_raw.copy()

df_mining_raw = pd.read_csv(r"../data/raw/icsid_cases_mining_202605.csv")
df_mining = df_mining_raw.copy()

Columns `casedecisions` and `caseproceedings` are in json object format.  Extract all the available data from these objects into a usuable dataframe.

In [4]:
# Extract data from the case proceeding and case decision dataframes and join to the main dataframe

import ast

df["casedecisions"] = df["casedecisions"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else None)
df["caseproceedings"] = df["caseproceedings"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else None)

case_decisions_col = pd.json_normalize(df["casedecisions"].explode()).reset_index(drop=True)
case_proceedings_col = pd.json_normalize(df["caseproceedings"].explode()).reset_index(drop=True)
    
df_expanded = pd.concat([df.drop(columns=["casedecisions", "caseproceedings"]), case_decisions_col, case_proceedings_col], axis=1)

# Clean up the large dataset by dropping columns with all null values and columns with all empty strings. Also, join the case proceeding and case decision data to the main dataframe.
df_clean = df_expanded.loc[:,
    ~(
        df_expanded.replace('',pd.NA)
        .isna()
        .all()
    )]

In [5]:
# Extract the messy/raw data of the country name from the `respondent` column and standardise it using the pycountry library.
from utils.country_utils import extract_country
df_clean[["country_clean", "country_iso3", "extraction_method"]] = df_clean["respondent"].apply(extract_country)

In [ ]:
print(f"Total number of cases: {df_clean.shape[0]}")
print(f"Total number of cases without a country name: {df_clean['country_clean'].isna().sum()}")

# Save dataframe of cases without a country name
df_cases_without_state = pd.Series(df_clean[df_clean["country_clean"].isna()][["caseno", "respondent"]].drop_duplicates().values.tolist())

# df_cases_without_state.to_csv(r"../data/interim/cases_without_state_icsid.csv", index=True) # Only uncomment if one wishes to save the interim dataset again.

Total number of cases: 1140
Total number of cases without a country name: 26


In [17]:
missing_countries = df_clean.loc[df_clean["country_clean"].isna()]

missing_cases = sorted(
    missing_countries["caseno"]
    .dropna()
    .unique()
)

print("=" * 60)
print("MISSING COUNTRY ENTRIES")
print("=" * 60)

if len(missing_cases) == 0:
    print("✓ No missing date entries found.")
else:
    print(f"Total number of cases missing a country: {len(missing_cases)}")
    for case in missing_cases:
        print(f"• {case}")

MISSING COUNTRY ENTRIES
Total number of cases missing a country: 26
• ARB/01/10
• ARB/05/12
• ARB/06/21
• ARB/07/3
• ARB/08/10
• ARB/10/11
• ARB/10/18
• ARB/10/20
• ARB/12/28
• ARB/13/24
• ARB/14/35
• ARB/19/18
• ARB/19/3
• ARB/20/50
• ARB/22/19
• ARB/22/4
• ARB/24/39
• ARB/25/12
• ARB/25/43
• ARB/25/44
• ARB/26/5
• ARB/76/1
• ARB/92/2
• ARB/98/8
• CONC(AF)/12/2
• CONC/19/1


In [23]:
# Manually add in the respondent countries per case.
ISCID_MANUAL_COUNTRY_CLEANUP = {
    "ARB/24/39": "Bangladesh",
    "ARB/22/4": "Bangladesh",
    "ARB/19/18": "Bangladesh",
    'ARB/10/11': "Bangladesh",
    "ARB/10/18": "Bangladesh",
    "ARB/92/2": "Bangladesh",
    "CONC/19/1": "Cameroon",
    "ARB/25/12": "Ecuador",
    "ARB/05/12": "Ecuador",
    "ARB/01/10": "Ecuador",
    "ARB/08/10": "Ecuador",
    "ARB/06/21": "Ecuador",
    "ARB/76/1": "France",
    "ARB/07/3": "Indonesia",
    "ARB/20/50": "Kosovo",
    "ARB/22/19": "Peru",
    "ARB/26/5": "Peru",
    "ARB/12/28": "Peru",
    "ARB/13/24": "Peru",
    "ARB/25/43": "Tanzania",
    "ARB/25/44": "Tanzania",
    "ARB/10/20": "Tanzania",
    "ARB/98/8": "Tanzania",
    "ARB/14/35": "Turkey",
    "ARB/19/3": "Rwanda",
    "CONC(AF)/12/2": "United States"
}

# Find rows in dataframe that need to be updated
mask = (
    df_clean["country_clean"].isna()
    &
    df_clean["caseno"]
    .isin(ISCID_MANUAL_COUNTRY_CLEANUP.keys())
)

# Update only missing dates
df_clean.loc[
    mask,
    "country_clean"
] = (
    df_clean.loc[
        mask,
        "caseno"
    ].map(ISCID_MANUAL_COUNTRY_CLEANUP)
)

df_clean["country_clean"].isna().sum()

np.int64(0)

In [ ]:
# Convert dates into datetime objects and check to see if there are any dud values.
df_clean["dateregistered"] = pd.to_datetime(df_clean["dateregistered"], errors="coerce")

parsed_dates = pd.to_datetime(
    df_clean["dateregistered"],
    format="%Y-%m-%d",
    errors="coerce"
)

bad_mask = (
    parsed_dates.isna() &
    df_clean["dateregistered"].notna()
)

missing_dates = df_clean.loc[df_clean["dateregistered"].isna()]
problem_rows = df_clean.loc[bad_mask][["caseno", "dateregistered"]]

C:\Users\User\AppData\Local\Temp\ipykernel_13572\3384676693.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean_countries["dateregistered"] = pd.to_datetime(df_clean_countries["dateregistered"], errors="coerce")


In [32]:
missing_cases = sorted(
    missing_dates["caseno"]
    .dropna()
    .unique()
)

problem_cases = sorted(
    problem_rows["caseno"]
    .dropna()
    .unique()
)

print("=" * 60)
print("MISSING DATE ENTRIES")
print("=" * 60)

if len(missing_cases) == 0:
    print("✓ No missing date entries found.")
else:
    for case in missing_cases:
        print(f"• {case}")

print("\n")

print("=" * 60)
print("UNPARSEABLE DATE ENTRIES")
print("=" * 60)

if len(problem_cases) == 0:
    print("✓ No unparseable date entries found.")
else:
    for case in problem_cases:
        print(f"• {case}")

MISSING DATE ENTRIES
• ADM/21/1
• UNCT/23/4


UNPARSEABLE DATE ENTRIES
✓ No unparseable date entries found.


Two cases in the data do not contain an entry for `dateregistered`.  Manually enter the date with reference to sources.

UNCT/23/4 - 30/11/2023 - https://icsid.worldbank.org/cases/case-database/case-detail?CaseNo=UNCT/23/4 

ADM/21/1 - 07/08/2020 - https://jusmundi.com/en/document/decision/en-asiaphos-limited-v-peoples-republic-of-china-composition-of-the-tribunal

In [ ]:
# Create manual dictionary of cases needed to be cleaned up
MANUAL_DATE_CLEANUP = {
    "UNCT/23/4": "2023-11-30",
    "ADM/21/1": "2020-08-07"
}

# Find rows in dataframe that need to be updated
mask = (
    df_clean["dateregistered"].isna()
    &
    df_clean["caseno"]
    .isin(MANUAL_DATE_CLEANUP.keys())
)

# Update only missing dates
df_clean.loc[
    mask,
    "dateregistered"
] = (
    df_clean.loc[
        mask,
        "caseno"
    ].map(MANUAL_DATE_CLEANUP)
)

# Convert to datetime
df_clean["dateregistered"] = pd.to_datetime(
    df_clean["dateregistered"],
    errors = "coerce"
)

In [ ]:
# Verify that cases have been cleaned up
df_clean.loc[df_clean["caseno"].isin(MANUAL_DATE_CLEANUP.keys())]

,caseid,caseno,claimant,isapproved,respondent,status,isapproved,dateregistered,isapproved,country_clean,country_iso3,extraction_method
142,146,UNCT/23/4,Alberta Petroleum Marketing Commission,Y,United States of America,Pending,Y,2023-11-30,Y,United States,USA,pycountry
202,207,ADM/21/1,AsiaPhos Limited and Norwest Chemicals Pte Ltd,Y,People’s Republic of China,Concluded,Y,2020-08-07,Y,China,CHN,embedded_pycountry


In [ ]:
# Extract the day, month, and year from the data column of the dataset
df_clean["dateregistered"] = pd.to_datetime(df_clean["dateregistered"], errors="coerce")
df_clean["registration_year"] = df_clean["dateregistered"].dt.year.astype("Int64")
df_clean["registration_month"] = df_clean["dateregistered"].dt.month.astype("Int64")
df_clean["registration_day"] = df_clean["dateregistered"].dt.day.astype("Int64")

In [ ]:
# Using the extracted filtered dataframe df_mining, mark which cases relate to the mining sector in the working dataframe
mining_cases = df_mining["caseno"]
df_clean["miningcase"] = df_clean["caseno"].isin(mining_cases).map({True: "Yes", False: "No"})

In [ ]:
# Save cleaned dataframe to csv file for documentation
df_clean.to_csv(r"../data/interim/icsid_clean_cases_202606_processed.csv")

### UNCTAD data